# 03. Chọn độ dài bước

Quét bước cố định quanh $1/L$, lưới dò tay không dùng $L$, và 18 cấu hình backtracking.
Kết quả vào chương 4 của báo cáo.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
pd.set_option("display.width", 160)

In [2]:
from src.dataset import load_processed, SWEEP
from src.experiment import BUILDERS, ExperimentGroup, run_group, summary_table
obj, X_test, y_test, cfg = load_processed("../data/processed", SWEEP)
print(f"n={obj.n:,} d={obj.d} L={obj.L:.4f} mu={obj.mu:.5f} kappa={obj.kappa:.1f}")

n=200,000 d=116 L=9.1156 mu=0.03407 kappa=267.5


In [3]:
groups = {}
for name in ("step-fixed", "step-blind", "step-armijo"):
    title, build = BUILDERS[name]
    groups[name] = run_group(ExperimentGroup(name, title, build), obj, "../results/raw")

step-fixed: loaded 8 runs from ../results/raw/step-fixed.json
step-blind: loaded 5 runs from ../results/raw/step-blind.json
step-armijo: loaded 18 runs from ../results/raw/step-armijo.json


In [4]:
df = pd.DataFrame(summary_table(groups["step-fixed"]))
df["bước"] = [r.step_label for r in groups["step-fixed"]]
df[["bước", "status", "iters_to_1e-06", "seconds_to_1e-06", "final_gap"]]

,bước,status,iters_to_1e-06,seconds_to_1e-06,final_gap
0,t = 2.1/L,diverged,NaN,NaN,5.669963e+06
1,t = 2/L,stalled,NaN,NaN,9.370342e-02
2,t = 1.9/L,converged,331.0,4.749504,3.474576e-18
3,t = 2/(L+mu),converged,766.0,10.873740,1.383410e-20
4,t = 1/L,converged,630.0,8.981211,3.520998e-18
5,t = 0.5/L,max_iter,1261.0,18.036011,1.624317e-13
6,t = 0.1/L,max_iter,NaN,NaN,5.348056e-06
7,t = 0.01/L,max_iter,NaN,NaN,1.330715e-02


Bước $1{,}9/L$ nhanh hơn bước tối ưu lý thuyết $2/(L+\mu)$. Phân tích modal dưới đây
giải thích vì sao: sai số ban đầu dồn vào các hướng trị riêng lớn.

In [5]:
ev, V = np.linalg.eigh(obj.hessian)
weight = 0.5 * ev * (V.T @ (0 - obj.w_star))**2
print(f"f(0) - f* = {weight.sum():.4f}")
print(f"  hướng lambda > 1   : {weight[ev > 1].sum()/weight.sum()*100:5.1f}% sai số ban đầu")
print(f"  hướng lambda < 0.1 : {weight[ev < 0.1].sum()/weight.sum()*100:5.1f}%")
for name, t in [("1.9/L", 1.9/obj.L), ("2/(L+mu)", 2/(obj.L+obj.mu))]:
    for k in (331, 766):
        print(f"  t={name:<9} k={k:>4}  gap du doan = {(weight*(1-t*ev)**(2*k)).sum():.3e}")

f(0) - f* = 5.5300
  hướng lambda > 1   :  80.5% sai số ban đầu
  hướng lambda < 0.1 :   0.3%
  t=1.9/L     k= 331  gap du doan = 9.799e-07
  t=1.9/L     k= 766  gap du doan = 6.822e-10
  t=2/(L+mu)  k= 331  gap du doan = 6.651e-04
  t=2/(L+mu)  k= 766  gap du doan = 9.953e-07


In [6]:
from src.figures import convergence_pair, cost_figure, save_figure
for name, recs in groups.items():
    show = [r for r in recs if "rho=0.5" in r.label] if name == "step-armijo" else recs
    convergence_pair(show, name, title=BUILDERS[name][0], out_dir="../results/figures")
save_figure(cost_figure([r for r in groups["step-armijo"] if "t0 = 1)" in r.label]),
            "step-armijo_cost", "../results/figures")

[PosixPath('../results/figures/step-armijo_cost.pdf'),
 PosixPath('../results/figures/step-armijo_cost.png')]